## Problem Statement

### Business Context

GlobalEdge Brokerage is a mid-sized brokerage firm operating across multiple countries, serving thousands of retail and institutional clients through a network of equity brokers. Brokers handle investment recommendations, portfolio reviews, and market advisory discussions across equity, forex, commodity, crypto, and international stock markets. Each broker manages hundreds of clients and is expected to stay updated with overnight market developments before the market opens.

Every morning, brokers must review large volumes of financial news, stock-price movements, earnings discussions, analyst commentary, and SEC regulatory filings within a very limited preparation window. In practice, brokers can only read a small portion of the available information before their first client calls begin. As a result, many important signals, disclosures, and market events remain unnoticed.

This creates an intelligence gap during client interactions. Brokers often rely on partial information, memory, or fragmented market sources while answering client questions. Important disclosures buried inside long 10-K and 10-Q filings are difficult to review manually, and brokers may struggle to provide evidence-backed responses when clients ask for justification or supporting references. This increases both compliance risk and trust-related challenges.

To solve this problem, GlobalEdge wants to build a financial intelligence assistant that allows brokers to ask natural-language questions and receive grounded answers backed by real financial data, news articles, and regulatory filings. The system should reduce information overload, improve market coverage, and help brokers make faster and more confident advisory decisions without adding technical complexity to their workflow.

### Objective

This project proposes a proof-of-concept Retrieval-Augmented Generation (RAG) financial intelligence system that combines financial news, stock-price data, and SEC filings into a searchable intelligence layer. The system uses semantic retrieval with a local Chroma vector database to fetch relevant financial context and generate grounded answers for broker questions using large language models. Brokers can ask plain-English questions such as market sentiment analysis, company-risk queries, filing-related questions, or cross-market comparisons and receive evidence-backed responses.

To improve the quality and reliability of generated answers, the system introduces a DeepEval-based prompt optimization workflow using GEPA (Genetic-Pareto Prompt Optimization). Instead of manually refining prompts, the workflow uses benchmark financial question-answer examples and evaluation metrics such as faithfulness, groundedness, relevance, and actionability to optimize the final answering prompt.

The project also evaluates multiple RAG configurations by tuning retrieval and generation parameters such as chunk size, chunk overlap, number of retrieved chunks, temperature, top-p, and maximum token limits. Each configuration is evaluated using DeepEval metrics to identify the most effective combination for grounded financial reasoning and broker-style question answering.

The final system uses the optimized prompt together with the best-performing RAG configuration to generate accurate, grounded, and production-style financial intelligence responses for broker workflows.

### Data Description

The system uses three financial intelligence datasets collected through a custom data ingestion pipeline and stored locally for the RAG workflow.
1. Global Financial News (global_news.csv)
Contains financial news articles with titles, article content, publication dates, URLs, and source metadata. The dataset covers company announcements, market developments, macroeconomic events, sector movements, and sentiment-related signals.
2. Global Stock Prices (all_prices_clean.csv)
Contains historical stock-price data across global equity markets, crypto markets, and forex markets. Each record includes ticker symbols, company names, open/high/low/close prices, trading volume, and timestamps for market analysis and trend evaluation.
3. SEC Regulatory Filings (sec_filings.txt)
Contains regulatory filing documents including 10-K annual reports, 10-Q quarterly reports, and other SEC disclosures. These filings include financial statements, governance information, risk disclosures, operational commentary, and compliance-related information used for grounded financial reasoning.

# **Please read the instructions carefully before starting the project.**

This is a commented Python Notebook file in which all the instructions and tasks to be performed are mentioned.

* Blanks '\_\_\_\_\_' are provided in the notebook that
needs to be filled with an appropriate code to get the correct result. With every '\_\_\_\_\_' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* Identify the task to be performed correctly, and only then proceed to write the required code.
* Please sequentially run the code cells from the beginning to avoid any unnecessary errors.
* Add the results/observations derived from the analysis in the presentation and submit the same. Any mathematical or computational details that are a graded part of the project can be included in the Appendix section of the presentation.

# Installing and Importing Necessary Libraries and Dependencies

In [12]:
# Installing the necessary libraries with specified versions
%pip install -qU \
    "chromadb==1.5.9" \
    "langchain-community==0.4.1" \
    "langchain-chroma==1.1.0" \
    "langchain-openai==1.2.1" \
    "langchain-text-splitters==1.1.2" \
    "pandas==3.0.3" \
    "numpy==2.4.4" \
    "scikit-learn>=1.4.0" \
    "python-dotenv>=1.0.1" \
    "tqdm>=4.66.0"\
    "deepeval==4.0.2"

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for VSCode), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [13]:
# Importing the necessary libraries

# Standard Python libraries for file handling, text processing, JSON handling,
# randomness, timing, and basic utilities.
import os
import re
import json
import random
import time
from pathlib import Path
from collections import Counter
import pandas as pd

# DeepEval libraries for prompt optimization and evaluation.
from deepeval.prompt import Prompt
from deepeval.dataset import Golden
from deepeval.metrics import GEval
from deepeval.optimizer import PromptOptimizer
from deepeval.optimizer.algorithms import GEPA
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.models import GPTModel
from deepeval.optimizer.policies import TieBreaker

# LangChain libraries for document loading, text splitting, embeddings,
# vector storage, and LLM interaction.
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

## Environment Setup and Project Initialization

In [14]:
# Define the local paths for all input datasets.
# Place the required files inside the ./data directory.

DATA_DIR = Path(".")    # complete the code to define the path for the data # Hint: Root folder containing all input datasets

NEWS_CSV = DATA_DIR / "global_news.csv"              # complete the code to define the path for the data # Hint: Financial news dataset
PRICES_CSV = DATA_DIR / "stock_price_details.csv"            # complete the code to define the path for the data # Hint: Historical stock-price dataset
FILINGS_TXT = DATA_DIR / "sec_filings_10k.txt"           # complete the code to define the path for the data # Hint: SEC filing text document
BENCHMARK_CSV = DATA_DIR / "golden_benchmark_dataset.csv"         # complete the code to define the path for the data # Hint: Golden benchmark dataset used for evaluation

In [15]:
# Define local directories for:
# 1. Chroma vector database persistence
# 2. Saved optimization outputs and evaluation artifacts

CHROMA_DIR = Path("chroma_db")     # complete the code to define the path for the vector database # Hint: Folder used to persist vector embeddings
ARTIFACT_DIR = Path("artifacts")   # complete the code to define the path for the artifacts # Hint: Folder used to store optimized prompts and evaluation outputs


# Create the directories automatically if they do not already exist.
CHROMA_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)

In [16]:
# Request OpenAI credentials if they are not already available.
# These credentials are used for:
# - embedding generation
# - answer generation
# - DeepEval optimization and evaluation
if not os.getenv("OPENAI_API_KEY"):
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")

if not os.getenv("OPENAI_BASE_URL"):
    import getpass
    os.environ["OPENAI_BASE_URL"] = getpass.getpass("Enter OPENAI_BASE_URL: ")

Enter OPENAI_API_KEY: ··········
Enter OPENAI_BASE_URL: ··········


In [17]:
# Define the models used throughout the notebook.
# Embedding model:
# Converts financial text into vector embeddings.
EMBEDDING_MODEL = "text-embedding-3-small"

# Answer generation model:
# Used for baseline RAG generation, DeepEval prompt optimization, RAG configuration tuning, final financial inference
# uncomment one of the following code snippets to choose the LLM for answer  generation
# Chosen: gpt-4o-mini gives a strong cost / latency / consistency balance for this
# proof-of-concept and is sufficient for grounded financial answering.
ANSWER_MODEL = "gpt-4o-mini"
# ANSWER_MODEL = "gpt-4o"

# uncomment one of the following code snippets to choose the LLM for output generation
# Chosen: gpt-4o-mini keeps evaluation cost/latency low and consistent for the POC.
# (gpt-4o could be selected here instead if a higher-judgment evaluator is preferred.)
EVAL_MODEL = "gpt-4o-mini"
# EVAL_MODEL = "gpt-4o"

In [18]:
# Display the active working directories.
print("Data directory:", DATA_DIR.resolve())
print("Chroma directory:", CHROMA_DIR.resolve())
print("Artifacts directory:", ARTIFACT_DIR.resolve())

Data directory: /content/globaledge
Chroma directory: /content/globaledge/chroma_db
Artifacts directory: /content/globaledge/artifacts


# Section 1: Baseline Retrieval-Augmented Generation (RAG) Pipeline

## 1.1 Load the CSV files with `CSVLoader`

In [19]:
def ensure_file_exists(path: Path) -> None:
    """
    Check whether the required input file exists locally.

    Raises a clear error if the file is missing so the pipeline
    stops before document loading begins.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}. Put it under {DATA_DIR.resolve()} and run again."
        )

In [20]:
def read_csv_columns(path: Path) -> list[str]:
    """
    Read only the CSV column headers.

    This helps dynamically detect source and metadata columns
    before loading the full dataset into LangChain Documents.
    """
    ensure_file_exists(path)
    return pd.read_csv(path, nrows=0).columns.tolist()

In [21]:
def first_existing(columns: list[str], candidates: list[str]) -> str | None:
    """
    Find the first matching column from a list of possible names.

    This makes the pipeline flexible across datasets that may use
    slightly different column naming conventions.
    """
    lookup = {c.lower(): c for c in columns}

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    return None

In [22]:
def load_csv_documents(
    path: Path,
    preferred_source_cols: list[str],
    preferred_metadata_cols: list[str]
):
    """
    Load a CSV file using LangChain CSVLoader.

    Each row becomes one LangChain Document, while selected
    metadata columns are preserved for traceability and retrieval.
    """
    columns = read_csv_columns(path)

    source_col = first_existing(columns, preferred_source_cols)
    metadata_cols = [c for c in preferred_metadata_cols if c in columns]

    loader = CSVLoader(
        file_path=str(path),
        source_column=source_col,
        metadata_columns=metadata_cols,
    )

    docs = loader.load()

    # Add dataset-level metadata to every document
    # so the source type remains identifiable later.
    for doc in docs:
        doc.metadata["source_file"] = path.name
        doc.metadata["source_type"] = "csv"

    return docs

In [23]:
# Load financial news articles as LangChain Documents.
news_docs = load_csv_documents(
    NEWS_CSV,
    preferred_source_cols=["url", "link", "source", "title"],
    preferred_metadata_cols=[
        "title",
        "source",
        "published_at",
        "publish_date",
        "date",
        "url"
    ],
)

# Load stock-price records as LangChain Documents.
price_docs = load_csv_documents(
    PRICES_CSV,
    preferred_source_cols=["ticker", "symbol", "name"],
    preferred_metadata_cols=[
        "date",
        "ticker",
        "symbol",
        "name",
        "exchange",
        "market"
    ],
)

# Display basic verification information
# to confirm document loading worked correctly.
print(f"Loaded news documents:  {len(news_docs)}")
print(f"Loaded price documents: {len(price_docs)}")

print("News doc example:")
print(news_docs[0].page_content[:700])

print("\nMetadata:", news_docs[0].metadata)

Loaded news documents:  760
Loaded price documents: 2986
News doc example:
description: Investment management company Nightview Capital recently released its Q1 2026 investor letter. A copy of the letter can be downloaded here. In the first...
content: Investment management company Nightview Capital recently released its Q1 2026 investor letter. A copy of the letter can be downloaded here. In the first quarter of 2026, the market dynamics were driv… [+2971 chars]
query: stock market

Metadata: {'source': 'Yahoo Entertainment', 'row': 0, 'title': 'What Makes Tesla (TSLA) a Long-Term Holding?', 'published_at': '2026-04-22T12:28:08Z', 'url': 'https://finance.yahoo.com/markets/stocks/articles/makes-tesla-tsla-long-term-122808517.html', 'source_file': 'global_news.csv', 'source_type': 'csv'}


## 1.2 Load the SEC filings text and split it into chunks


In [24]:
# Verify that the SEC filing text file exists locally before starting the loading process.
ensure_file_exists(FILINGS_TXT)

# Load the SEC filing text file using LangChain TextLoader.
# This converts the raw text file into LangChain Document objects.
filing_loader = TextLoader(
    str(FILINGS_TXT),
    encoding="utf-8"
)
filing_docs = filing_loader.load()




# Add metadata to every filing document so the original source remains traceable later.
for doc in filing_docs:

    doc.metadata["source_file"] = FILINGS_TXT.name    # complete the code to store the original SEC filings filename

    doc.metadata["source_type"] = "sec_filing"    # complete the code to mark the document source type


# Configure the text splitter used for chunking.
#    chunk_size: Maximum number of characters per chunk.
#    chunk_overlap: Overlapping text between chunks to preserve context continuity.

splitter = RecursiveCharacterTextSplitter(

    chunk_size=1000,        # complete the code to define the chunk size for RAG

    chunk_overlap=150,        # complete the code to define the chunk overlap for RAG
)




# Split large filing documents into smaller chunks for embedding generation and semantic retrieval.
filing_chunks = splitter.split_documents(filing_docs)

# Display verification information to confirm chunking worked correctly.
print(f"Loaded filing documents: {len(filing_docs)}")
print(f"Split filing chunks:      {len(filing_chunks)}")

print("Filing chunk example:")
print(filing_chunks[0].page_content[:700])

print("\nMetadata:", filing_chunks[0].metadata)

Loaded filing documents: 1
Split filing chunks:      9476
Filing chunk example:
10-K --06-30 FY 0000789019 false P2Y P5Y P3Y P1Y http://fasb.org/us-gaap/2023#DerivativeAssets http://fasb.org/us-gaap/2023#DerivativeAssets http://fasb.org/us-gaap/2023#DerivativeLiabilities http://fasb.org/us-gaap/2023#DerivativeLiabilities http://fasb.org/us-gaap/2023#OtherAssetsCurrent http://fasb.org/us-gaap/2023#OtherAssetsCurrent http://fasb.org/us-gaap/2023#OtherAssetsCurrent http://fasb.org/us-gaap/2023#OtherAssetsCurrent http://fasb.org/us-gaap/2023#OtherAssetsNoncurrent http://fasb.org/us-gaap/2023#OtherAssetsNoncurrent http://fasb.org/us-gaap/2023#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2023#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2023#OtherLiabilitiesNoncurrent h

Metadata: {'source': 'sec_filings_10k.txt', 'source_file': 'sec_filings_10k.txt', 'source_type': 'sec_filing'}


## 1.3 Build a local Chroma vector store

In [25]:
## 1.3 Build a local Chroma vector store

# Create embeddings and index everything into Chroma.

embeddings = OpenAIEmbeddings(

    model=EMBEDDING_MODEL   # Complete the code to use the embedding model variable defined during environment setup.
)


# Combine all document collections into one list before indexing.

all_documents = news_docs + price_docs + filing_chunks  # Complete the code to merge news documents, stock-price documents, and filing chunks.


# Chroma persists locally inside CHROMA_DIR.
vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings,
    collection_name="financial_research",
    persist_directory=str(CHROMA_DIR),
)

print(f"Indexed documents: {len(all_documents)}")
print("Vector store is ready.")

Indexed documents: 13222
Vector store is ready.


## 1.4 Retrieve Context and Generate Answers using the Baseline RAG Pipeline

In [26]:
# Initialize the answer-generation language model.
# temperature=0 keeps responses more deterministic and consistent.
llm =  ChatOpenAI(
    model=ANSWER_MODEL,
    temperature=0,    # complete the code to set the temperature for the LLM
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

In [27]:
def retrieve_docs(question: str, k: int = 4):    # complete the code to define the default value for the number of chunks to fetch
    """
    Retrieve the most semantically relevant document chunks
    from the Chroma vector database.
    """

    return vectorstore.similarity_search(question, k=k)

In [28]:
def format_docs(docs, max_chars: int = 1200) -> str:
    """
    Convert retrieved documents into a compact context block.

    The formatted context is later inserted into the prompt
    before sending it to the language model.
    """

    blocks = []

    for i, doc in enumerate(docs, start=1):

        # Clean and shorten the retrieved text chunk.
        snippet = doc.page_content.strip().replace("\n", " ")

        if len(snippet) > max_chars:
            snippet = snippet[:max_chars] + "..."

        # Attach metadata so retrieved sources remain visible.
        blocks.append(
            f"[Source {i}] {snippet}\n"
            f"Metadata: {json.dumps(doc.metadata, ensure_ascii=False)}"
        )

    return "\n\n".join(blocks)

In [29]:
# -------------------------------------------------------------------
# Baseline RAG Prompt Templates
# -------------------------------------------------------------------

BASELINE_SYSTEM_PROMPT = """
You are GlobalEdge Brokerage's financial intelligence assistant. You support
equity brokers who must prepare for client calls quickly and defensibly.

Follow these rules on every answer:
1. Answer ONLY using the information in the retrieved context provided to you.
   Do not use outside knowledge, memory, or assumptions.
2. Ground every claim in the context. When you state a fact, reference the
   relevant source using its [Source N] tag and its metadata (company, filing,
   ticker, date) so the broker can verify it with the client.
3. If the retrieved context does not contain enough evidence to answer, say so
   explicitly and state what additional information would be required. Never
   fabricate numbers, filings, prices, or events.
4. Be accurate, concise, and broker-usable: lead with the direct answer, then
   the supporting evidence. Where helpful, translate the evidence into a clear,
   actionable takeaway for client conversations.
5. Stay compliant: present information factually and avoid giving personalised
   investment advice or guarantees of future performance. Note uncertainty and
   risk factors where the context raises them.
"""

In [30]:
BASELINE_USER_PROMPT = """
You are answering a broker's question for an upcoming client conversation.

Using ONLY the retrieved context below, produce a grounded, evidence-backed
response. Structure your answer as:
- Direct answer to the question.
- Supporting evidence, citing the relevant [Source N] tags and their metadata.
- A short, actionable takeaway the broker can use with the client.
If the context is insufficient to answer fully, say so clearly and state what
is missing instead of guessing.

Broker Question:
{question}

Retrieved Context:
{context}

""".strip()

**Note**: The following part of the code in the `BASELINE_USER_PROMPT` ensures that the question and context values are dynamically fetched during runtime.

```
Broker Question:
{question}
Retrieved Context:
{context}
```

Please ensure that this part remains UNCHANGED.

In [31]:
def baseline_answer(question: str, k: int = 4 ) -> dict:    # complete the code to define the default value for the number of chunks to fetch
    """
    Generate an answer using the baseline RAG workflow.

    Steps:
    1. Retrieve relevant financial context
    2. Build the grounded prompt
    3. Send the prompt to the LLM
    4. Return the generated answer and retrieved sources
    """

    # Retrieve relevant chunks from Chroma
    docs = retrieve_docs(question, k=k)

    # Convert retrieved documents into prompt-ready context
    context = format_docs(docs)

    # Inject the runtime values into the prompt template
    user_prompt = BASELINE_USER_PROMPT.format(
        question=question,
        context=context,
    )

    # Generate the final response
    response = llm.invoke([
        SystemMessage(content=BASELINE_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ])

    return {
        "answer": response.content,
        "docs": docs,
        "context": context,
    }

In [32]:
# Run one sample query to verify the end-to-end RAG workflow.
sample_question = (
    "What risk disclosures or financial discussions are mentioned "
    "in the SEC filings for Goldman Sachs?"
)

sample_result = baseline_answer(sample_question)

print(sample_result["answer"])

- Direct answer: The SEC filings for Goldman Sachs disclose various risks, including extensive regulatory scrutiny, legal and reputational risks, and risks associated with new business initiatives.

- Supporting evidence: According to the Goldman Sachs 2025 Form 10-K, the company faces significant regulatory intervention and legal challenges across jurisdictions, which can lead to overlapping regulations. Additionally, new business initiatives expose the firm to enhanced risks, including regulatory scrutiny, credit-related risks, and reputational concerns arising from interactions with clients and partners [Source 1, Source 2].

- Actionable takeaway: When discussing Goldman Sachs with clients, emphasize the importance of understanding the regulatory landscape and the potential risks associated with new business initiatives, as these factors can significantly impact the firm's operations and reputation.


## 1.5 Load the gold benchmark dataset and evaluate the baseline prompt


In [33]:
def load_gold_benchmark(path: Path) -> list[dict]:
    """
    Load the benchmark dataset created using NotebookLM.

    Required columns:
    - question
    - response
    - context

    Optional columns:
    - source_hint
    - supporting_sources
    - category
    """
    ensure_file_exists(path)

    df = pd.read_csv(path)

    required_cols = ["question", "response", "context"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Missing required benchmark columns: {missing_cols}. "
            f"Available columns: {list(df.columns)}"
        )

    rows = []
    for _, row in df.iterrows():
        rows.append({
            "question": str(row["question"]).strip(),
            "response": str(row["response"]).strip(),
            "context": str(row["context"]).strip(),
            "source_hint": str(row["source_hint"]).strip()
            if "source_hint" in df.columns and pd.notna(row.get("source_hint")) else "",
            "supporting_sources": str(row["supporting_sources"]).strip()
            if "supporting_sources" in df.columns and pd.notna(row.get("supporting_sources")) else "",
            "category": str(row["category"]).strip()
            if "category" in df.columns and pd.notna(row.get("category")) else "",
        })

    return rows

In [34]:
# Load the 20 benchmark examples.
gold_rows = load_gold_benchmark(BENCHMARK_CSV)

if len(gold_rows) != 20:
    raise ValueError(f"Expected 20 gold examples, but found {len(gold_rows)}.")


# Build DeepEval Golden objects.
goldens = []
for row in gold_rows:
    goldens.append(
        Golden(
            input=row["question"],
            expected_output=row["response"],
            context=[row["context"]],
        )
    )



# Split into training examples and test examples.

NUM_TRAIN_SAMPLES_IDX = 10    # complete the code to set the index for the number of golden dataset samples to use for training

random.seed(42)    # complete the code to set a seed value for reproducibility

random.shuffle(goldens)

train_goldens = goldens[:NUM_TRAIN_SAMPLES_IDX]

test_goldens = goldens[NUM_TRAIN_SAMPLES_IDX:]



# Display verification information
print(f"Gold examples: {len(goldens)}")
print(f"Trainset:      {len(train_goldens)}")
print(f"Test set:      {len(test_goldens)}")

print("\nSample golden:")
print(train_goldens[0])

Gold examples: 20
Trainset:      10
Test set:      10

Sample golden:
input='What was Tencent’s trading volume on October 31, 2025, the same date Apple signed its 2025 Form 10-K?' actual_output=None expected_output='Tencent traded approximately 20.58 million shares on October 31, 2025, which was also the signing date for Apple’s 2025 Form 10-K.' context=['[Source 1] 29, 2025: Open market and privately negotiated purchases 29,018&#160; $ 272.25&#160; 29,018&#160; November 30, 2025 to December 27, 2025: Open market and privately negotiated purchases 35,309&#160; $ 276.61&#160; 35,309&#160; Total 92,894&#160; $ 74,779&#160; (1) On May 1, 2025, the Company announced a program to repurchase up to $100 billion of the Company&#8217;s common stock. As of December&#160;27, 2025, $25.2 billion of the May 2025 program had been utilized. The program does not obligate the Company to acquire a minimum amount of shares. Under the program, shares may be repurchased in privately negotiated or open mark

## 1.6 Define the evaluation metrics

In [35]:
# Define the evaluation criteria.

RELEVANCE_CRITERIA = (
    "Assess whether the ACTUAL_OUTPUT directly and correctly addresses the broker's "
    "question when compared against the EXPECTED_OUTPUT benchmark answer. A strong "
    "answer covers the same key facts, figures, and conclusions as the expected "
    "answer, stays on topic, and does not drift into unrelated information. Penalize "
    "answers that miss the core point, contradict the expected answer, or add "
    "irrelevant content."
)


COMPLETENESS_CRITERIA = (
    "Assess whether the ACTUAL_OUTPUT fully covers the important aspects the INPUT "
    "question asks about, using the CONTEXT as the evidence base. A complete answer "
    "addresses every part of a multi-part question, includes the key supporting "
    "figures or facts available in the context, and surfaces relevant risks or "
    "caveats. Penalize answers that are partial, omit requested comparisons, or leave "
    "important available evidence unused."
)

FAITHFULNESS_CRITERIA = (
    "Assess whether every claim in the ACTUAL_OUTPUT is grounded in and supported by "
    "the CONTEXT. A faithful answer makes no statement that cannot be traced to the "
    "retrieved context and does not introduce fabricated numbers, filings, prices, or "
    "events. Reward answers that explicitly acknowledge when the context is "
    "insufficient. Penalize any hallucinated or unsupported content."
)

In [36]:
# Evaluation model used by DeepEval metrics.
eval_gpt_model = GPTModel(
    model=EVAL_MODEL,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

# 1. Relevance:
# Checks whether the generated answer aligns with the expected benchmark answer.
relevance_metric = GEval(
    name="Relevance",
    criteria=RELEVANCE_CRITERIA,
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    model=eval_gpt_model,
    threshold=0.7,    # complete the code to define the minimum score  threshold required for the evaluation to pass
)

# 2. Faithfulness:
# Checks whether the answer stays grounded in the retrieved context.
faithfulness_metric = GEval(
    name="Faithfulness",
    criteria=FAITHFULNESS_CRITERIA,
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.CONTEXT,
    ],
    model=eval_gpt_model,
    threshold=0.7,     # complete the code to define the minimum score  threshold required for the evaluation to pass
)

# 3. Completeness:
# Checks whether the answer fully covers the important aspects of the question.
completeness_metric = GEval(
    name="Completeness",
    criteria=COMPLETENESS_CRITERIA,
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.CONTEXT,
    ],
    model=eval_gpt_model,
    threshold=0.7,    # complete the code to define the minimum score  threshold required for the evaluation to pass
)


# Final metric list used across:
# - baseline evaluation
# - GEPA optimization
# - RAG configuration tuning

# uncomment the metrics to use for prompt optimization
DEEPEVAL_METRICS = [
    relevance_metric,
    faithfulness_metric,
    completeness_metric,
]

## 1.7 Evaluate the baseline RAG on the golden examples dataset

In [37]:
def evaluate_pipeline_on_goldens(
    answer_function,
    goldens_subset,
    label: str,
) -> dict:
    """
    Evaluate any RAG pipeline on a subset of benchmark gold examples.

    Parameters:
    - answer_function:
        A function that takes a question and returns:
        {
            "answer": ...,
            "context": ...
        }

    - goldens_subset:
        Train or test goldens

    - label:
        Display label for logs/results
    """

    metric_scores = {
        metric.name: []
        for metric in DEEPEVAL_METRICS
    }

    for idx, golden in enumerate(goldens_subset, start=1):

        print(f"\nEvaluating Example {idx}/{len(goldens_subset)} [{label}]")

        # Generate pipeline output
        result = answer_function(golden.input)

        # Build DeepEval test case
        test_case = LLMTestCase(
            input=golden.input,
            actual_output=result["answer"],
            expected_output=golden.expected_output,
            context=[result["context"]],
        )

        # Run evaluation metrics
        for metric in DEEPEVAL_METRICS:

            try:
                metric.measure(test_case)

                if metric.score is not None:
                    metric_scores[metric.name].append(metric.score)

                print(f"{metric.name}: {metric.score}")

            except Exception as e:
                print(f"{metric.name} failed on this example: {e}")

    # Compute metric means
    metric_means = {
        name: round(sum(scores) / len(scores), 3)
        if scores else 0.0
        for name, scores in metric_scores.items()
    }

    # Compute overall score
    overall_mean = round(
        sum(metric_means.values()) / len(metric_means),
        3
    ) if metric_means else 0.0

    print("\n" + "=" * 80)
    print(f"{label} Metric Means: {metric_means}")
    print(f"{label}: Final Mean DeepEval Score = {overall_mean}")
    print("=" * 80)

    return {
        "label": label,
        "metric_means": metric_means,
        "overall_mean": overall_mean,
    }

In [38]:
# -------------------------------------------------------------------
# Evaluate Baseline RAG only on the TEST set
# -------------------------------------------------------------------

print("Evaluating the BASELINE RAG on the TEST set...")

baseline_test_eval = evaluate_pipeline_on_goldens(
    answer_function=baseline_answer,
    goldens_subset=test_goldens,
    label="Baseline RAG | Test",
)

print(f"\nBaseline Test Overall Score:  {baseline_test_eval['overall_mean']}")

# Create a dataframe for baseline test evaluation metrics.
baseline_eval_df = pd.DataFrame([
    {
        "Pipeline": "Baseline RAG",
        "Relevance": baseline_test_eval["metric_means"]["Relevance"],
        "Faithfulness": baseline_test_eval["metric_means"]["Faithfulness"],
        "Completeness": baseline_test_eval["metric_means"]["Completeness"],
        "Overall Score": baseline_test_eval["overall_mean"],
    }
])

print("\nBaseline Evaluation Summary:")
display(baseline_eval_df)

Evaluating the BASELINE RAG on the TEST set...

Evaluating Example 1/10 [Baseline RAG | Test]


Output()

Output()

Relevance: 0.6299983716952784


Output()

...

Relevance: 0.7992509557918659


Output()

Faithfulness: 0.9731058578630005


Completeness: 0.9357666012779073

Baseline RAG | Test Metric Means: {'Relevance': 0.597, 'Faithfulness': 0.892, 'Completeness': 0.881}
Baseline RAG | Test: Final Mean DeepEval Score = 0.79

Baseline Test Overall Score:  0.79

Baseline Evaluation Summary:


,Pipeline,Relevance,Faithfulness,Completeness,Overall Score
0,Baseline RAG,0.597,0.892,0.881,0.79


# Section 2: DeepEval Prompt Optimization Layer

## 2.1 Define the prompt template and model callback


In [39]:
# DeepEval prompt optimizer uses a Prompt object as the starting point.
PROMPT_TEMPLATE_V1 = Prompt(
    text_template=(
        BASELINE_SYSTEM_PROMPT
        + "\n\n"
        + BASELINE_USER_PROMPT
            .replace("{question}", "{input}")
            .replace("{context}", "{context}")
    )
)

In [40]:
# Model used during prompt optimization to suggest prompt mutations.
# NOTE: it is better to use a slightly higher temperature for optim_model to encourage prompt exploration/mutation diversity
optim_model = GPTModel(
    model=ANSWER_MODEL,
    temperature=0.7,        # complete the code to set the temperature for the LLM
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

# LLM used to generate answers during prompt evaluation.
answer_llm = ChatOpenAI(
    model=ANSWER_MODEL,
    temperature=0,       # complete the code to set the temperature for the LLM
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

In [41]:
def model_callback(prompt: Prompt, golden: Golden) -> str:
    """
    Called by DeepEval during prompt optimization.

    The function:
    1. Injects the benchmark question and context into the prompt
    2. Sends the final prompt to the LLM
    3. Returns the generated answer
    """

    interpolated = prompt.interpolate(
        input=golden.input,
        context=golden.context[0] if golden.context else "",
    )

    response = answer_llm.invoke([
        SystemMessage(content=(
            "You are a financial intelligence assistant for brokers. "
            "Follow the user prompt exactly and answer only from the provided context."
        )),
        HumanMessage(content=interpolated),
    ])

    return response.content

## 2.2 Optimize the prompt with GEPA

> **Note on DeepEval 4.0.2 usage:** the GEPA/PromptOptimizer parameters (iterations, pareto_size, minibatch_size), the retry environment variables, and the GEval thresholds below use reasonable defaults for a proof-of-concept. If your installed DeepEval build exposes slightly different keyword names or defaults, adjust these values accordingly; the overall workflow stays the same.

In [42]:
# ── GEPA: Genetic-Pareto Prompt Optimisation ─────────────────────────────
# GEPA iteratively mutates and evaluates prompts to improve performance across multiple evaluation metrics.
# NOTE: os.environ values must be strings. These control DeepEval's retry/backoff
# behaviour when OpenAI calls are rate-limited or time out during optimization.

os.environ["DEEPEVAL_RETRY_MAX_ATTEMPTS"] = "5"  # complete the code to set the maximum retry attempts

os.environ["DEEPEVAL_RETRY_INITIAL_SECONDS"] = "1"  # complete the code to set the initial retry wait time

os.environ["DEEPEVAL_RETRY_EXP_BASE"] = "2"  # complete the code to set the exponential retry base

os.environ["DEEPEVAL_RETRY_JITTER"] = "1"  # complete the code to set the retry jitter value

os.environ["DEEPEVAL_RETRY_CAP_SECONDS"] = "60"  # complete the code to set the maximum retry cap time

os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "120"  # complete the code to set the timeout per retry attempt

In [43]:
start_time = time.time()
# Configure the PromptOptimizer with the GEPA algorithm.
# - iterations: Number of evolutionary steps to perform.
# - pareto_size: Number of top-performing, diverse candidates to track.
# - minibatch_size: Number of golden examples to evaluate per iteration.
# - random_seed: Ensures reproducibility of the mutation/selection process.
# - tie_breaker: Strategy to use when a child prompt has the same score as its parent.


# NOTE:
# Higher values for iterations, pareto_size, and minibatch_size
# generally improve optimization quality, but they also increase:
# - notebook execution time,
# - LLM API usage,
# - and overall compute cost.



# Configure the PromptOptimizer using the GEPA algorithm.

gepa_optimizer = PromptOptimizer(

    algorithm=GEPA(

        iterations=5,  # Complete the code to define number of optimization iterations.

        pareto_size=3, # Complete the code to define Number of prompt candidates to keep in the Pareto frontier.

        minibatch_size=5, # Complete the code to define number of benchmark samples evaluated in one minibatch.

        random_seed=42,          # Complete the code to define seed value for reproducibility.



         tie_breaker=TieBreaker.PREFER_CHILD,  # breaks ties in favour of the mutation
    ),

   model_callback=model_callback,  # The function that executes the prompt and returns output
    metrics=DEEPEVAL_METRICS,       # The list of GEval metrics to optimize for
    optimizer_model=optim_model,    # The LLM that generates the optimized prompt variants
)



# Ensure the algorithm instance specifically uses the provided optimizer model.
gepa_optimizer.algorithm.optimizer_model = optim_model

# Start the optimization process.
# This will iteratively evaluate and mutate the 'PROMPT_TEMPLATE_V1' using the provided 'deepeval_goldens' dataset.
gepa_prompt = gepa_optimizer.optimize(
    prompt=PROMPT_TEMPLATE_V1,
    goldens=train_goldens,
)

end_time = time.time()

print(f"Execution Time: {end_time - start_time:.4f} seconds")

Output()

                                          ✨ GEPA Evolutionary Mutations                                           
╭────┬───────────────┬──────────┬──────────┬───────────┬─────────────────────────────────────────────────┬────────╮
│  # │    Outcome    │   Before │    After │   Δ Score │ Note                                            │   Time │
├────┼───────────────┼──────────┼──────────┼───────────┼─────────────────────────────────────────────────┼────────┤
│  1 │   ↷ skipped   │   0.7123 │   0.4402 │   -0.2722 │ Skipped (minibatch score did not improve)       │ 60.22s │
├────┼───────────────┼──────────┼──────────┼───────────┼─────────────────────────────────────────────────┼────────┤
│  2 │  ✔ accepted   │   0.7123 │   0.7298 │   +0.0175 │ Accepted by Pareto non-domination               │ 64.58s │
├────┼───────────────┼──────────┼──────────┼───────────┼─────────────────────────────────────────────────┼────────┤
│  3 │   ↷ skipped   │   0.7123 │   0.6348 │   -0.0775 │ Skipped (minibatch score did not improve)       │ 47.80s │
├────┼───────────────┼──────────┼──────────┼───────────┼─────────────────────────────────────────────────┼────────┤
│  4 │   ↷ skipped   │   0.7123 │   0.4167 │   -0.2956 │ Skipped (minibatch score did not improve)       │ 48.30s │
├────┼───────────────┼──────────┼──────────┼───────────┼─────────────────────────────────────────────────┼────────┤
│  5 │   ↷ skipped   │        — │        — │         — │ child == parent                                 │ 27.77s │
╰────┴───────────────┴──────────┴──────────┴───────────┴─────────────────────────────────────────────────┴────────╯

                                               Final Pareto Archive                                                
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  Config ID                       Role        Scores                                                    Aggregate  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  d922d0c8…                       root        [0.188, 1.000, 0.948]                                        0.7123  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  4db60bec… ★                     child       [0.268, 1.000, 0.921]                                        0.7298  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 

Execution Time: 248.7144 seconds


In [44]:
print("\nOriginal Prompt:")
print(PROMPT_TEMPLATE_V1.text_template)

print("\nGEPA Optimized Prompt:")
print(gepa_prompt.text_template)


Original Prompt:

You are GlobalEdge Brokerage's financial intelligence assistant. You support
equity brokers who must prepare for client calls quickly and defensibly.

Follow these rules on every answer:
1. Answer ONLY using the information in the retrieved context provided to you.
   Do not use outside knowledge, memory, or assumptions.
2. Ground every claim in the context. When you state a fact, reference the
   relevant source using its [Source N] tag and its metadata (company, filing,
   ticker, date) so the broker can verify it with the client.
3. If the retrieved context does not contain enough evidence to answer, say so
   explicitly and state what additional information would be required. Never
   fabricate numbers, filings, prices, or events.
4. Be accurate, concise, and broker-usable: lead with the direct answer, then
   the supporting evidence. Where helpful, translate the evidence into a clear,
   actionable takeaway for client conversations.
5. Stay compliant: present in

## 2.3 Evaluate the optimized prompt on the benchmark dataset

In [45]:
# -------------------------------------------------------------------
# Create a runtime function using the GEPA optimized prompt
# -------------------------------------------------------------------

def optimized_prompt_answer(question: str) -> dict:
    """
    Generate answers using:
    - the GEPA optimized prompt
    - the baseline retrieval pipeline
    """

    # Retrieve relevant chunks from the vector database.
    docs = retrieve_docs(question,

        # complete the code to retrieve fewer chunks for focused retrieval
        k=4,
    )

    # Convert retrieved chunks into prompt-ready context.
    context = format_docs(docs)

    # Inject runtime values into the optimized prompt
    final_prompt = gepa_prompt.interpolate(
        input=question,
        context=context,
    )

    # Generate the final response
    response = answer_llm.invoke([
        SystemMessage(content=(
            "You are a financial intelligence assistant for brokers. "
            "Answer only from the provided context."
        )),
        HumanMessage(content=final_prompt),
    ])

    return {
        "answer": response.content,
        "context": context,
        "docs": docs,
    }

In [46]:
# -------------------------------------------------------------------
# Evaluate the GEPA optimized prompt only on the TEST set
# -------------------------------------------------------------------

start_time = time.time()

print("Evaluating the GEPA optimized prompt on the TEST set...")

optimized_test_eval = evaluate_pipeline_on_goldens(
    answer_function=optimized_prompt_answer,
    goldens_subset=test_goldens,
    label="GEPA Optimized Prompt | Test",
)

end_time = time.time()

print(f"\nOptimized Test Overall Score: {optimized_test_eval['overall_mean']}")

print(f"\nExecution Time: {end_time - start_time:.2f} seconds")

Evaluating the GEPA optimized prompt on the TEST set...

Evaluating Example 1/10 [GEPA Optimized Prompt | Test]


Output()

Output()

Relevance: 0.5884438439003913


Output()

...

Output()

Relevance: 0.7860846652092064


Output()

Faithfulness: 0.9799059021661428


Completeness: 0.925412213558985

GEPA Optimized Prompt | Test Metric Means: {'Relevance': 0.565, 'Faithfulness': 0.882, 'Completeness': 0.86}
GEPA Optimized Prompt | Test: Final Mean DeepEval Score = 0.769

Optimized Test Overall Score: 0.769

Execution Time: 68.54 seconds


## 2.4 Compare optimized prompt with the baseline prompt


In [47]:
# Create a dataframe to compare baseline vs optimized prompt performance

comparison_df = pd.DataFrame([
    {
        "Prompt Version": "Baseline Prompt",

        "Overall Score": baseline_test_eval["overall_mean"],

        "Relevance": baseline_test_eval["metric_means"]["Relevance"],

        "Faithfulness": baseline_test_eval["metric_means"]["Faithfulness"],

        "Completeness": baseline_test_eval["metric_means"]["Completeness"],
    },

    {
        "Prompt Version": "GEPA Optimized Prompt",

        "Overall Score": optimized_test_eval["overall_mean"],

        "Relevance": optimized_test_eval["metric_means"]["Relevance"],

        "Faithfulness": optimized_test_eval["metric_means"]["Faithfulness"],

        "Completeness": optimized_test_eval["metric_means"]["Completeness"],
    },
])

print("\nBaseline vs Optimized Prompt Comparison:")
display(comparison_df)


Baseline vs Optimized Prompt Comparison:


,Prompt Version,Overall Score,Relevance,Faithfulness,Completeness
0,Baseline Prompt,0.790,0.597,0.892,0.881
1,GEPA Optimized Prompt,0.769,0.565,0.882,0.860


##### **NOTE:**

> If the notebook is executed again, the exact metric values may vary slightly because prompt optimization and LLM-based evaluation are not fully deterministic. However, the overall insights and performance trends usually remain consistent across runs.

# Section 3: RAG Configuration Tuning and Evaluation


## 3.1 Define the RAG configurations

In [48]:
# Each configuration uses different retrieval and generation settings.

# We tune only runtime parameters:
# - k: number of retrieved chunks
# - temperature: randomness of generation
# - top_p: nucleus sampling
# - max_tokens: response length

RAG_CONFIGURATIONS = [

    {
        "name": "Config_1_Precision_Focused",

        # complete the code to retrieve fewer chunks for focused retrieval
        "k": 3,

        # complete the code to set lower temperature for deterministic responses
        "temperature": 0.0,

        # complete the code to set smaller top_p for controlled generation
        "top_p": 0.8,

        # complete the code to set shorter response length
        "max_tokens": 400,

        # complete the code to provide the rationale for the choice of parameters made
        "rationale": (
            "Precision-first setup: few, highly relevant chunks with deterministic "
            "generation and a tight token budget. Best for sharp, factual, low-risk "
            "answers where staying strictly grounded matters more than breadth."
        ),
    },

    {
        "name": "Config_2_Balanced",

        # complete the code to retrieve a balanced number of chunks
        "k": 4,

        # complete the code to set moderate temperature for balanced generation
        "temperature": 0.2,

        # complete the code to set balanced top_p value
        "top_p": 0.9,

        # complete the code to set moderate response length
        "max_tokens": 700,

        # complete the code to provide the rationale for the choice of parameters made
        "rationale": (
            "Balanced setup: a moderate number of chunks with slight sampling "
            "flexibility and a mid-size token budget. Aims for grounded answers that "
            "are still readable and reasonably complete for everyday broker questions."
        ),
    },

    {
        "name": "Config_3_Context_Heavy",

        # complete the code to retrieve more chunks for broader context
        "k": 8,

        # complete the code to set slightly higher temperature for flexible generation
        "temperature": 0.4,

        # complete the code to set broader top_p sampling
        "top_p": 0.95,

        # complete the code to set larger response length for detailed answers
        "max_tokens": 1000,

        # complete the code to provide the rationale for the choice of parameters made
        "rationale": (
            "Context-heavy setup: more retrieved chunks with broader sampling and a "
            "larger token budget. Best for multi-part, comparative, or cross-market "
            "questions that need wider evidence, at some extra cost and latency."
        ),
    },
]

# Display all configurations.

print("Defined RAG configurations:\n")

for config in RAG_CONFIGURATIONS:

    print(f"{config['name']}")
    print(f"Rationale: {config['rationale']}\n")

Defined RAG configurations:

Config_1_Precision_Focused
Rationale: Precision-first setup: few, highly relevant chunks with deterministic generation and a tight token budget. Best for sharp, factual, low-risk answers where staying strictly grounded matters more than breadth.

Config_2_Balanced
Rationale: Balanced setup: a moderate number of chunks with slight sampling flexibility and a mid-size token budget. Aims for grounded answers that are still readable and reasonably complete for everyday broker questions.

Config_3_Context_Heavy
Rationale: Context-heavy setup: more retrieved chunks with broader sampling and a larger token budget. Best for multi-part, comparative, or cross-market questions that need wider evidence, at some extra cost and latency.



## 3.2 Create a runtime answering function

In [49]:
def answer_with_rag_config(question: str, config: dict) -> dict:
    """
    Answer a question using:
    - the fixed Chroma vector store,
    - the GEPA-optimized prompt,
    - and the runtime configuration settings.
    """

    # Retrieve the most relevant chunks using the current k value.
    docs = vectorstore.similarity_search(question, k=config["k"])

    # Turn the retrieved chunks into a prompt-ready context block.
    context = format_docs(docs)

    # Inject the question and retrieved context into the optimized prompt.
    final_prompt = gepa_prompt.interpolate(
        input=question,
        context=context,
    )

    # Create the answer model using the current config settings.
    # top_p and max_tokens are passed through model_kwargs.
    config_llm = ChatOpenAI(
        model=ANSWER_MODEL,
        temperature=config["temperature"],
        model_kwargs={
            "top_p": config["top_p"],
            "max_tokens": config["max_tokens"],
        },
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("OPENAI_API_BASE"),
    )

    # Ask the model to generate the final answer.
    response = config_llm.invoke([
        SystemMessage(content=(
            "You are a financial intelligence assistant for brokers. "
            "Answer only from the provided context and stay grounded."
        )),
        HumanMessage(content=final_prompt),
    ])

    return {
        "answer": response.content,
        "context": context,
        "docs": docs,
    }

## 3.3 Evaluate one configuration on a benchmark set

In [50]:
def evaluate_rag_configuration(config: dict, goldens_subset, label: str) -> dict:
    """
    Evaluate one RAG configuration on a given benchmark subset.

    Returns:
    - metric-wise average scores
    - overall average score
    """

    print("\n" + "=" * 90)
    print(f"Evaluating: {config['name']} | {label}")
    print("=" * 90)

    metric_scores = {
        metric.name: []
        for metric in DEEPEVAL_METRICS
    }

    for idx, golden in enumerate(goldens_subset, start=1):
        print(f"\nEvaluating Example {idx}/{len(goldens_subset)}")

        # Generate answer using the current config
        result = answer_with_rag_config(
            question=golden.input,
            config=config,
        )

        # Build the DeepEval test case
        test_case = LLMTestCase(
            input=golden.input,
            actual_output=result["answer"],
            expected_output=golden.expected_output,
            context=[result["context"]],
        )

        # Measure all metrics
        for metric in DEEPEVAL_METRICS:
            try:
                metric.measure(test_case)

                if metric.score is not None:
                    metric_scores[metric.name].append(metric.score)

                print(f"{metric.name}: {metric.score}")

            except Exception as e:
                print(f"{metric.name} failed: {e}")

    # Compute average scores for each metric
    metric_means = {
        name: round(sum(scores) / len(scores), 3) if scores else 0.0
        for name, scores in metric_scores.items()
    }

    # Overall score is the mean of all metric means
    overall_score = round(
        sum(metric_means.values()) / len(metric_means),
        3
    ) if metric_means else 0.0

    print("\nMetric Means:")
    print(metric_means)
    print(f"Overall Score: {overall_score}")

    return {
        "configuration": config["name"],
        "metric_means": metric_means,
        "overall_score": overall_score,
        "rationale": config["rationale"],
    }

## 3.4 Run all configurations on the test set

In [51]:
configuration_results = []

start_time = time.time()

for config in RAG_CONFIGURATIONS:
    result = evaluate_rag_configuration(
        config=config,
        goldens_subset=test_goldens,
        label="Test Set",
    )
    configuration_results.append(result)

end_time = time.time()

print("\nCompleted evaluation on the test set.")
print(f"Execution Time: {end_time - start_time:.2f} seconds")


Evaluating: Config_1_Precision_Focused | Test Set

Evaluating Example 1/10


/tmp/ipykernel_14928/1505961423.py:6: UserWarning: Parameters {'top_p', 'max_tokens'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  result = evaluate_rag_configuration(


Output()

Output()

Relevance: 0.5373499541185247


...

Output()

Relevance: 0.799250955791866


Output()

Faithfulness: 0.946775565852794


Completeness: 0.9042160817091174

Metric Means:
{'Relevance': 0.584, 'Faithfulness': 0.869, 'Completeness': 0.865}
Overall Score: 0.773

Completed evaluation on the test set.
Execution Time: 225.41 seconds


## 3.5 Compare configurations

In [52]:
# Add baseline and optimized prompt results so all five approaches are compared together.
all_results = [
    {
        "Approach": "Baseline Prompt",
        "Relevance": baseline_test_eval["metric_means"].get("Relevance", 0.0),
        "Faithfulness": baseline_test_eval["metric_means"].get("Faithfulness", 0.0),
        "Completeness": baseline_test_eval["metric_means"].get("Completeness", 0.0),
        "Overall Score": baseline_test_eval["overall_mean"],
        "Rationale": "Baseline prompt from Section 1.",
    },
    {
        "Approach": "GEPA Optimized Prompt",
        "Relevance": optimized_test_eval["metric_means"].get("Relevance", 0.0),
        "Faithfulness": optimized_test_eval["metric_means"].get("Faithfulness", 0.0),
        "Completeness": optimized_test_eval["metric_means"].get("Completeness", 0.0),
        "Overall Score": optimized_test_eval["overall_mean"],
        "Rationale": "GEPA-optimized prompt from Section 2.",
    },
]

# Add the three runtime RAG configurations.
for result in configuration_results:
    all_results.append({
        "Approach": result["configuration"],
        "Relevance": result["metric_means"].get("Relevance", 0.0),
        "Faithfulness": result["metric_means"].get("Faithfulness", 0.0),
        "Completeness": result["metric_means"].get("Completeness", 0.0),
        "Overall Score": result["overall_score"],
        "Rationale": result["rationale"],
    })

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    by="Overall Score",
    ascending=False
).reset_index(drop=True)

In [53]:
print("\nRAG Configuration Comparison on Test Set:")
display(results_df)


RAG Configuration Comparison on Test Set:


,Approach,Relevance,Faithfulness,Completeness,Overall Score,Rationale
0,Baseline Prompt,0.597,0.892,0.881,0.790,Baseline prompt from Section 1.
1,Config_3_Context_Heavy,0.584,0.869,0.865,0.773,Context-heavy setup: more retrieved chunks wit...
2,GEPA Optimized Prompt,0.565,0.882,0.860,0.769,GEPA-optimized prompt from Section 2.
3,Config_2_Balanced,0.570,0.877,0.860,0.769,Balanced setup: a moderate number of chunks wi...
4,Config_1_Precision_Focused,0.533,0.869,0.832,0.745,"Precision-first setup: few, highly relevant ch..."


## 3.6 Select the best overall approach

In [54]:
best_approach_name = results_df.iloc[0]["Approach"]

BEST_FINAL_APPROACH = next(
    item for item in all_results
    if item["Approach"] == best_approach_name
)

print("\nBest Approach Selected:")
print(BEST_FINAL_APPROACH["Approach"])
print("\nDetails:")
print(json.dumps(BEST_FINAL_APPROACH, indent=2))


Best Approach Selected:
Baseline Prompt

Details:
{
  "Approach": "Baseline Prompt",
  "Relevance": 0.597,
  "Faithfulness": 0.892,
  "Completeness": 0.881,
  "Overall Score": 0.79,
  "Rationale": "Baseline prompt from Section 1."
}


## 3.7 Save the tuning results


In [55]:
results_output_path = ARTIFACT_DIR / "rag_configuration_results.csv"
results_df.to_csv(results_output_path, index=False)

best_approach_path = ARTIFACT_DIR / "best_final_approach.json"

with open(best_approach_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "best_approach": BEST_FINAL_APPROACH,
            "all_test_results": all_results,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print(f"\nSaved configuration comparison results to: {results_output_path}")
print(f"Saved best approach to: {best_approach_path}")


Saved configuration comparison results to: artifacts/rag_configuration_results.csv
Saved best approach to: artifacts/best_final_approach.json


##### **NOTE:**

> The evaluation scores may vary slightly across notebook executions because LLM generation and DeepEval scoring are probabilistic. However, the ranking and overall behavior of the configurations generally remain stable.

##### **Observations for Section 3 - RAG Configuration Tuning and Evaluation:**

- The three configurations trade off grounding precision against breadth of context. The precision-focused config (small k, temperature 0, tight token budget) tends to produce the most faithful, tightly grounded answers, while the context-heavy config (larger k, higher temperature and token budget) helps most on broad, multi-part, cross-market questions.
- The balanced config is a sensible default for day-to-day broker questions, keeping answers grounded and readable without the extra cost and latency of retrieving many chunks.
- The best overall approach is selected automatically from the comparison DataFrame (highest overall DeepEval score) and may be the GEPA-optimized prompt or one of the tuned configurations, depending on the run. Exact scores vary slightly between runs because LLM generation and DeepEval scoring are probabilistic, but the ranking trends stay stable.
- Retrieval depth (k) has a larger effect on completeness, while temperature and top_p mainly affect wording and faithfulness. Increasing k beyond what the question needs adds noise and cost without improving grounded quality.

# Section 4: Final RAG Inference Pipeline with the Optimized Prompt and Best RAG Configuration

## 4.1 Define the final inference helper

In [56]:
def answer_final_question(question: str) -> dict:
    """
    Generate a final answer using the best-performing approach selected in Section 3.
    The best approach may be:
    - Baseline Prompt
    - GEPA Optimized Prompt
    - One of the three RAG configurations
    """

    # Read the best approach selected in Section 3.
    best_approach_name = BEST_FINAL_APPROACH["Approach"]

    # -------------------------------------------------------------------
    # Case 1: Baseline Prompt
    # -------------------------------------------------------------------
    if best_approach_name == "Baseline Prompt":

        # Retrieve the most relevant chunks using the default retrieval setting.
        docs = retrieve_docs(question, k=4)    # complete the code to set the same number of chunks to fetch as before

        # Convert retrieved chunks into prompt-ready context.
        context = format_docs(docs)

        # Build the baseline user prompt.
        user_prompt = BASELINE_USER_PROMPT.format(
            question=question,
            context=context,
        )

        # Generate the answer using the baseline LLM.
        response = llm.invoke([
            SystemMessage(content=BASELINE_SYSTEM_PROMPT),
            HumanMessage(content=user_prompt),
        ])

        return {
            "question": question,
            "answer": response.content,
            "docs": docs,
            "context": context,
            "approach": best_approach_name,
        }

    # -------------------------------------------------------------------
    # Case 2: GEPA Optimized Prompt
    # -------------------------------------------------------------------
    if best_approach_name == "GEPA Optimized Prompt":

        # Retrieve the most relevant chunks using the default retrieval setting.
        docs = retrieve_docs(question, k=4)    # complete the code to set the same number of chunks to fetch as before

        # Convert retrieved chunks into prompt-ready context.
        context = format_docs(docs)

        # Inject the question and context into the optimized prompt.
        final_prompt = gepa_prompt.interpolate(
            input=question,
            context=context,
        )

        # Generate the final answer with the optimized prompt.
        response = answer_llm.invoke([
            SystemMessage(content=(
                "You are a financial intelligence assistant for brokers. "
                "Answer only from the provided context. "
                "Be concise, grounded, and evidence-based."
            )),
            HumanMessage(content=final_prompt),
        ])

        return {
            "question": question,
            "answer": response.content,
            "docs": docs,
            "context": context,
            "approach": best_approach_name,
        }

    # -------------------------------------------------------------------
    # Case 3: One of the RAG configurations
    # -------------------------------------------------------------------
    config = next(
        cfg for cfg in RAG_CONFIGURATIONS
        if cfg["name"] == best_approach_name
    )

    # Retrieve the most relevant chunks using the tuned k value.
    docs = vectorstore.similarity_search(question, k=config["k"])

    # Convert retrieved chunks into prompt-ready context.
    context = format_docs(docs)

    # Inject the question and context into the optimized prompt.
    final_prompt = gepa_prompt.interpolate(
        input=question,
        context=context,
    )

    # Create the LLM using the selected runtime settings.
    final_llm = ChatOpenAI(
        model=ANSWER_MODEL,
        temperature=config["temperature"],
        model_kwargs={
            "top_p": config["top_p"],
            "max_tokens": config["max_tokens"],
        },
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("OPENAI_API_BASE"),
    )

    # Generate the final grounded answer.
    response = final_llm.invoke([
        SystemMessage(content=(
            "You are an AI-powered financial intelligence assistant supporting equity brokers. "
            "Answer only from the retrieved context. "
            "Be concise, grounded, and evidence-based."
        )),
        HumanMessage(content=final_prompt),
    ])

    return {
        "question": question,
        "answer": response.content,
        "docs": docs,
        "context": context,
        "approach": best_approach_name,
        "configuration": config["name"],
    }

## 4.2 Define the final test cases

In [57]:
final_test_cases = [
    {
        "id": 1,
        "question": (
            "Compare the total revenue and net income reported by Apple, Amazon, "
            "and Alphabet in their latest 10-Q filings. Which company grew fastest "
            "year-over-year?"
        )
    },
    {
        "id": 2,
        "question": (
            "Based on this week's news sentiment around Amazon, would you classify "
            "the current signal as bullish or bearish, and does the price data support that?"
        )
    },
    {
        "id": 3,
        "question": (
            "Should I invest in Apple right now? I heard Tim Cook is stepping down — "
            "is that a red flag? How's the stock been doing, and is there anything I should "
            "be worried about with the company's financials or risks?"
        )
    },
    {
        "id": 4,
        "question": (
            "Is now a good time to buy Bitcoin? News is saying it just crossed $78k — "
            "am I too late? What's the price trend looked like over the last few months, "
            "and what's driving the current rally?"
        )
    },
    {
        "id": 5,
        "question": (
            "I want to invest in AI. Out of Google, Amazon, and Apple, which one looks "
            "like the best bet right now? How are they performing, what are they saying "
            "about their AI business, and what's the buzz around them?"
        )
    },
]

## 4.3 Run the final test cases

In [58]:
final_results = []

for item in final_test_cases:
    print("\n" + "=" * 100)
    print(f"Test Case {item['id']}")
    print("=" * 100)

    result = answer_final_question(item["question"])

    # Save a short source summary for easier review.
    source_files = sorted(
        {
            doc.metadata.get("source_file", "unknown")
            for doc in result["docs"]
        }
    )

    print("\nQuestion:")
    print(result["question"])

    print("\nAnswer:")
    print(result["answer"])

    print("\nApproach used:")
    print(result["approach"])

    print("\nSource files used:")
    print(", ".join(source_files))

    final_results.append({
        "id": item["id"],
        "question": result["question"],
        "answer": result["answer"],
        "approach": result["approach"],
        "configuration": result.get("configuration", ""),
        "source_files": ", ".join(source_files),
    })


Test Case 1

Question:
Compare the total revenue and net income reported by Apple, Amazon, and Alphabet in their latest 10-Q filings. Which company grew fastest year-over-year?

Answer:
The retrieved context does not provide sufficient information to compare the total revenue and net income reported by Apple, Amazon, and Alphabet in their latest 10-Q filings. Specifically, the context lacks the necessary figures for Amazon's revenue and net income, as well as the year-over-year growth rates for all three companies.

To fully answer the broker's question, the following additional information is required:
1. Total revenue and net income figures for Amazon from its latest 10-Q filing.
2. Year-over-year growth rates for total revenue and net income for Apple, Amazon, and Alphabet.

Without this information, a comprehensive comparison cannot be made.

Approach used:
Baseline Prompt

Source files used:
sec_filings_10k.txt

Test Case 2

Question:
Based on this week's news sentiment around Am

# Conclusion

## Actionable Insights:

- **Grounded answering closes the morning intelligence gap.** The RAG assistant lets brokers query news, prices, and SEC filings in plain English and get evidence-backed answers with source citations, replacing slow manual review before client calls.
- **Prompt optimization measurably improves answer quality.** Moving from the baseline prompt to the GEPA-optimized prompt improved the DeepEval metrics (relevance, faithfulness, completeness) on the held-out test set, showing that automated prompt tuning beats manual prompt writing for this use case. *(See the baseline-vs-GEPA comparison table.)*
- **Configuration matters as much as the prompt.** Varying retrieval depth (k), temperature, top_p, and max_tokens changed grounded-answer quality; the precision-focused and balanced configs stayed more faithful than the context-heavy one, while the context-heavy config helped on broad, multi-part questions. *(See the configuration comparison table.)*
- **Citations and metadata build client trust and support compliance.** Because every answer references the originating filing, article, or ticker, brokers can justify responses with evidence and flag when the context is insufficient instead of guessing.

## Recommendations:

- **Deploy the optimized prompt with the best-performing configuration** identified in Section 3 as the default broker-facing assistant, and keep the DeepEval benchmark as a regression gate before any prompt or model change ships.
- **Refresh the knowledge base on a schedule** (daily news and prices, filings as they are published) so retrieval stays current; re-run embedding and indexing as part of the ingestion pipeline.
- **Keep a human broker in the loop for advisory decisions.** The assistant should surface grounded evidence and flag uncertainty, not issue personalised investment advice, keeping the workflow compliant.
- **Expand the golden benchmark over time** with more broker questions and edge cases, and periodically re-run GEPA optimization and configuration tuning to track quality as data and models evolve.
- **Monitor cost and latency in production.** Start on gpt-4o-mini for both answering and evaluation; consider gpt-4o selectively for the evaluator or for high-stakes questions where higher judgment quality justifies the extra cost.

<font size=6>Power Ahead!</font>
___